# Symbolic CSP network on MNIST (gradient-free)

A fully **symbolic, gradient-free** image classifier built with
`tessera.experimental.csp_sr` — no genetic programming, no gradient descent.

```
image (14x14)
  |  4 channels {image, |dx|, |dy|, Laplacian}   (symbolic feature layer)
  |  7x7 region pooling
  v
196 features --> 10 sparse symbolic class heads (csp_sr, one-vs-rest) --> argmax -> digit
```

- **Feature layer**: channels + region pooling — a fixed symbolic
  representation of the image.
- **Classifier**: for each digit, csp_sr enumerates symbolic forms over the
  196 features and fits a sparse per-class scorer by closed-form least
  squares.
- Each class head is an explicit, inspectable expression; the whole model
  is a layered graph (visualized in section 3).

Colab-runnable; the feature-matrix build uses the jit'd opcode-tape
interpreter on GPU (`use_jax`).

## 1. Setup

In [ ]:
# Cell 1.1 - Setup: clone/refresh tessera to LATEST, install, verify.
import os, sys, subprocess
REPO_DIR = 'tessera'
REPO_URL = 'https://github.com/davechendatascience/tessera.git'

def _run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    return r.returncode, r.stdout, r.stderr

if not os.path.exists(REPO_DIR):
    print('Cloning tessera...'); _run(['git', 'clone', REPO_URL])
else:
    print('Refreshing tessera to origin/main...')
    _run(['git', '-C', REPO_DIR, 'fetch', '--depth', '1', 'origin', 'main'])
    _run(['git', '-C', REPO_DIR, 'reset', '--hard', 'origin/main'])
_, sha, _ = _run(['git', '-C', REPO_DIR, 'log', '-1', '--format=%h %s'])
print('tessera @', sha.strip())

print('Installing tessera[benchmarks,jax]...')
rc, out, err = _run([sys.executable, '-m', 'pip', 'install', '-e',
                     f'./{REPO_DIR}[benchmarks,jax]'])
if rc != 0:
    print('pip stderr tail:', err.splitlines()[-8:])
src_abs = os.path.abspath(os.path.join(REPO_DIR, 'src'))
if src_abs not in sys.path:
    sys.path.insert(0, src_abs)

import importlib
try:
    import tessera.experimental.csp_sr as _csp; importlib.reload(_csp)
    assert hasattr(_csp, 'discover')
    import tessera.experimental.symbolic_interp as _si   # jit'd interpreter (GPU)
    print('OK: csp_sr + symbolic_interp loaded.')
except Exception as e:
    print(f'LOAD FAILED ({e}). Restart runtime & re-run this cell.')

In [ ]:
# Cell 1.2 - Imports.
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from tessera.experimental.csp_sr import discover, CSPSRConfig, expr_to_str
from tessera.expression.tree import evaluate as eval_tree
print('csp_sr loaded')

import jax
print(f'JAX devices: {jax.devices()}')
_on_gpu = any('cuda' in str(d).lower() or 'gpu' in str(d).lower() for d in jax.devices())
# use_jax -> build the feature matrix via the jit'd opcode-tape interpreter
# (compile once) on GPU. On CPU numpy is faster, so default off there.
USE_INTERPRETER = _on_gpu
print('  -> GPU: use_jax=True (jit interpreter Phi-build).' if _on_gpu else
      '  -> CPU: use_jax=False (numpy is faster on CPU).')

## 2. Load MNIST (all 10 digits)

Fast load of the compact ~11MB `mnist.npz` (the file Keras uses) -- no ARFF
parsing, no TensorFlow import. 200 train + 100 test per class, downsampled
28x28 -> 14x14.

In [ ]:
# Cell 2 - Fast MNIST load + 10-class data (14x14).
SEED = 2026
N_PER_TRAIN = 200
N_PER_TEST  = 100
IMG_SIZE = 14

def _load_mnist():
    """Fetch the compact mnist.npz and np.load it (~1-2s, no ARFF/TF).
    Falls back to sklearn fetch_openml if the download is blocked."""
    import os, urllib.request
    url = 'https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz'
    path = 'mnist.npz'
    try:
        if not os.path.exists(path):
            print('  downloading mnist.npz (~11MB, one-time)...')
            urllib.request.urlretrieve(url, path)
        with np.load(path) as dd:
            Xa = np.concatenate([dd['x_train'], dd['x_test']]).astype(np.float32) / 255.0
            ya = np.concatenate([dd['y_train'], dd['y_test']]).astype(int)
        return Xa, ya, 'npz'
    except Exception as e:
        print(f'  npz failed ({e}); using sklearn openml...')
        from sklearn.datasets import fetch_openml
        m = fetch_openml('mnist_784', version=1, as_frame=False, cache=True)
        return (m.data.reshape(-1, 28, 28).astype(np.float32) / 255.0,
                m.target.astype(int), 'openml')

if '_MNIST' not in globals():
    print('Loading MNIST...'); t0 = time.time(); _MNIST = _load_mnist()
    print(f'  done in {time.time()-t0:.1f}s (via {_MNIST[2]})')
X, y, _ = _MNIST

rng = np.random.default_rng(SEED)
tr_list, te_list, ytr, yte = [], [], [], []
for d in range(10):
    idx = rng.permutation(np.where(y == d)[0])
    tr_list.append(idx[:N_PER_TRAIN])
    te_list.append(idx[N_PER_TRAIN:N_PER_TRAIN + N_PER_TEST])
    ytr += [d] * N_PER_TRAIN; yte += [d] * N_PER_TEST
tr_idx = np.concatenate(tr_list); te_idx = np.concatenate(te_list)
labels_train = np.array(ytr); labels_test = np.array(yte)

# batched 2x downsample (one reshape + mean)
imgs_train = X[tr_idx].reshape(-1, 14, 2, 14, 2).mean(axis=(2, 4))
imgs_test  = X[te_idx].reshape(-1, 14, 2, 14, 2).mean(axis=(2, 4))
print(f'TRAIN {imgs_train.shape}, TEST {imgs_test.shape}  '
      f'(10 classes x {N_PER_TRAIN}/{N_PER_TEST})')

## 3. Symbolic feature layer + csp_sr classifier

The symbolic feature layer (4 channels x 7x7 region pooling) gives 196
features; csp_sr fits a sparse symbolic scorer per digit (one-vs-rest), and
we classify by argmax. Gradient-free, fast, and every head is an explicit
expression. Section 3.2 prints the network; 3.3 draws it as a layer graph.

In [ ]:
# Cell 3.1 - Symbolic feature layer + csp_sr per-class scorers.
def make_channel_features(imgs, grid=7):
    """4 channels {image, |dx|, |dy|, laplacian} region-pooled to grid x grid.
    Returns (features (N,F), names)."""
    N, H, W = imgs.shape
    ch = {'image': imgs}
    gx = np.zeros_like(imgs); gx[:, :, 1:] = np.abs(imgs[:, :, 1:] - imgs[:, :, :-1]); ch['gx'] = gx
    gy = np.zeros_like(imgs); gy[:, 1:, :] = np.abs(imgs[:, 1:, :] - imgs[:, :-1, :]); ch['gy'] = gy
    lap = np.zeros_like(imgs)
    lap[:, 1:-1, 1:-1] = (imgs[:, :-2, 1:-1] + imgs[:, 2:, 1:-1]
                          + imgs[:, 1:-1, :-2] + imgs[:, 1:-1, 2:]
                          - 4 * imgs[:, 1:-1, 1:-1])
    ch['lap'] = np.abs(lap)
    bh, bw = H // grid, W // grid
    feats, names = [], []
    for cname, c in ch.items():
        pooled = (c[:, :grid * bh, :grid * bw]
                  .reshape(N, grid, bh, grid, bw).mean(axis=(2, 4))
                  .reshape(N, grid * grid))
        for r in range(grid * grid):
            feats.append(pooled[:, r]); names.append(f'{cname}{r}')
    return np.stack(feats, axis=1).astype(np.float64), names

GRID = 7
F_train, feat_names = make_channel_features(imgs_train, GRID)
F_test,  _          = make_channel_features(imgs_test,  GRID)
print(f'Symbolic feature layer: {len(feat_names)} features (4 channels x {GRID}x{GRID})')

env_train = {feat_names[i]: F_train[:, i] for i in range(len(feat_names))}
env_test  = {feat_names[i]: F_test[:, i]  for i in range(len(feat_names))}

# poly_degree=1 -> sparse symbolic LINEAR head per class (STLSQ). Try
# poly_degree=2 for nonlinear feature-product terms (small trees -> the
# jit interpreter helps most there on GPU).
csp_cfg = CSPSRConfig(poly_degree=1, stlsq_threshold=0.03, max_terms=60,
                      use_jax=USE_INTERPRETER)
print('Fitting 10 per-class csp_sr scorers (one-vs-rest, gradient-free)...')
t0 = time.time()
csp_scorers = []
for d in range(10):
    res = discover(env_train, 2.0 * (labels_train == d) - 1.0, csp_cfg)
    csp_scorers.append(res)
    print(f'  digit {d}: {res.n_terms} terms, train R2={res.r2:.3f}')

def csp_scores(env):
    return np.stack([np.asarray(eval_tree(s.expr, env), dtype=np.float64)
                     for s in csp_scorers], axis=1)
csp_tr_acc = float((np.argmax(csp_scores(env_train), axis=1) == labels_train).mean())
csp_preds  = np.argmax(csp_scores(env_test), axis=1)
csp_te_acc = float((csp_preds == labels_test).mean())
print(f'\nCSP classifier: TRAIN={csp_tr_acc:.4f}  TEST={csp_te_acc:.4f}  '
      f'(fit {time.time()-t0:.1f}s, gradient-free, use_jax={USE_INTERPRETER})')

In [ ]:
# Cell 3.2 - PRINT THE NETWORK (feature layer + class heads).
def print_csp_network(scorers, feat_names, grid, top=6):
    print('=' * 72)
    print('CSP SYMBOLIC NETWORK   (feature layer + 10 symbolic class heads)')
    print('=' * 72)
    print(f'Feature layer:  4 channels {{image, |dx|, |dy|, laplacian}}')
    print(f'                x {grid}x{grid} region pooling  =  {len(feat_names)} features')
    print(f'Classifier:     10 per-class scorers via csp_sr (one-vs-rest,')
    print(f'                gradient-free closed-form least squares)')
    print('-' * 72)
    for d, s in enumerate(scorers):
        terms = sorted(s.terms, key=lambda t: -abs(t[0]))[:top]
        body = '  '.join(f'{c:+.2f}*{k.split(":")[-1]}' for c, k in terms)
        more = '' if len(s.terms) <= top else f'  (+{len(s.terms) - top} more)'
        print(f'  score[{d}] = {s.intercept:+.2f}  {body}{more}')
    print('=' * 72)
    print('  predict(image) = argmax_d  score[d]( feature_layer(image) )')
    print(f'  TRAIN acc = {csp_tr_acc:.4f}    TEST acc = {csp_te_acc:.4f}')

print_csp_network(csp_scorers, feat_names, GRID)

In [ ]:
# Cell 3.3 - Architecture diagram (DL-style layer graph).
def visualize_csp_network(scorers, feat_names, grid, test_acc=None, figsize=(15, 8)):
    """image -> channels -> region pooling -> N features -> 10 sparse
    symbolic class heads -> argmax. Each head is annotated with its top
    symbolic term (blue +, red -), so the sparse structure is visible."""
    n_ch = 4; chans = ['image', '|dx|', '|dy|', 'lap']
    n_feat = len(feat_names); n_cls = len(scorers)
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_xlim(0, 16); ax.set_ylim(0, 11); ax.axis('off')
    for txt, xc in [('Input', 1.0), ('Channels', 3.6), ('Region pool', 6.5),
                    ('Symbolic class heads', 10.4), ('Predict', 14.7)]:
        ax.text(xc, 10.5, txt, ha='center', fontsize=12, weight='bold')
    ax.add_patch(mpatches.FancyBboxPatch((0.3, 5.0), 1.4, 1.0,
                 boxstyle='round,pad=0.05', fc='#FFE4B5', ec='black', lw=1.4))
    ax.text(1.0, 5.5, 'image\n14x14', ha='center', va='center', fontsize=10, weight='bold')
    cy = np.linspace(8.6, 2.4, n_ch); cx = 2.9
    for nm, yy in zip(chans, cy):
        ax.add_patch(mpatches.FancyBboxPatch((cx, yy - 0.38), 1.5, 0.76,
                     boxstyle='round,pad=0.04', fc='#B0E0E6', ec='black', lw=1.1))
        ax.text(cx + 0.75, yy, nm, ha='center', va='center', fontsize=10)
        ax.annotate('', xy=(cx, yy), xytext=(1.7, 5.5),
                    arrowprops=dict(arrowstyle='->', color='gray', lw=1))
    fx = 5.8
    ax.add_patch(mpatches.FancyBboxPatch((fx, 2.0), 1.5, 7.0,
                 boxstyle='round,pad=0.05', fc='#E6E6FA', ec='black', lw=1.4))
    ax.text(fx + 0.75, 5.5, f'7x7 pool\n-> {n_feat}\nfeatures', ha='center',
            va='center', fontsize=10, weight='bold', rotation=90)
    for yy in cy:
        ax.annotate('', xy=(fx, 5.5), xytext=(cx + 1.5, yy),
                    arrowprops=dict(arrowstyle='->', color='gray', lw=0.8))
    clx = 9.6; cly = np.linspace(9.7, 1.3, n_cls)
    for d, yy in enumerate(cly):
        ax.add_patch(mpatches.Circle((clx, yy), 0.30, fc='#90EE90', ec='black', lw=1.1))
        ax.text(clx, yy, str(d), ha='center', va='center', fontsize=10, weight='bold')
        ax.annotate('', xy=(clx - 0.30, yy), xytext=(fx + 1.5, 5.5),
                    arrowprops=dict(arrowstyle='->', color='gray', lw=0.4, alpha=0.45))
        terms = sorted(scorers[d].terms, key=lambda t: -abs(t[0]))
        if terms:
            coef, key = terms[0]; feat = key.split(':')[-1]
            col = '#0066CC' if coef > 0 else '#CC0000'
            extra = f'  (+{len(terms) - 1})' if len(terms) > 1 else ''
            ax.text(clx + 0.5, yy, f'{coef:+.0f}*{feat}{extra}', ha='left',
                    va='center', fontsize=8, color=col, family='monospace')
    ax.add_patch(mpatches.FancyBboxPatch((13.9, 5.0), 1.7, 1.0,
                 boxstyle='round,pad=0.05', fc='#DDA0DD', ec='black', lw=1.4))
    ax.text(14.75, 5.5, 'argmax\n-> digit', ha='center', va='center', fontsize=10, weight='bold')
    for yy in cly:
        ax.annotate('', xy=(13.9, 5.5), xytext=(clx + 0.30, yy),
                    arrowprops=dict(arrowstyle='->', color='gray', lw=0.4, alpha=0.4))
    acc = '' if test_acc is None else f' - TEST acc {test_acc:.3f}'
    fig.suptitle(f'CSP symbolic network: {n_ch} channels x 7x7 pool = {n_feat} '
                 f'features -> {n_cls} sparse symbolic heads (gradient-free){acc}',
                 fontsize=12, weight='bold', y=0.98)
    plt.tight_layout(); return fig

visualize_csp_network(csp_scorers, feat_names, GRID, test_acc=csp_te_acc)
plt.show()

In [ ]:
# Cell 3.4 - Confusion matrix + per-digit accuracy.
from sklearn.metrics import confusion_matrix as _cm_fn
cm = _cm_fn(labels_test, csp_preds)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax = axes[0]
im = ax.imshow(cm, cmap='Greens')
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title(f'CSP classifier confusion (TEST {csp_te_acc:.3f})')
thr = cm.max() / 2
for i in range(10):
    for j in range(10):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > thr else 'black', fontsize=8)
plt.colorbar(im, ax=ax, label='count')
ax = axes[1]
pda = cm.diagonal() / cm.sum(axis=1)
bars = ax.bar(range(10), pda, color='C2')
for bar, acc in zip(bars, pda):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{acc:.2f}', ha='center', fontsize=8)
ax.axhline(csp_te_acc, color='red', ls='--', alpha=0.7, label=f'overall = {csp_te_acc:.3f}')
ax.axhline(0.10, color='gray', ls=':', alpha=0.5, label='chance (0.10)')
ax.set_xticks(range(10)); ax.set_ylim(0, 1.05)
ax.set_xlabel('digit'); ax.set_ylabel('per-class TEST accuracy')
ax.set_title('CSP per-digit accuracy'); ax.legend(loc='lower right', fontsize=9)
plt.tight_layout(); plt.show()

## What's next - honest scope

A fast, fully symbolic, gradient-free classifier (~0.91 on 14x14 MNIST,
~0.94 at full resolution), with each class an inspectable expression.

- **Higher MNIST accuracy**: richer feature layer (finer pooling, more
  channels), `poly_degree=2` (nonlinear feature-product terms), or stacked
  csp_sr (`discover_boosted`). A linear symbolic head over pooled channels
  tops out around ~0.92-0.95; pushing to **0.99** essentially needs LEARNED
  convolutional features (a CNN at ~0.99) -- representation learning, which
  is backprop territory that csp_sr (no cross-layer credit) cannot do alone.
- **ImageNet** is out of reach for this architecture: 224x224 / 1000 classes
  need deep learned features. A symbolic linear head over hand-pooled
  channels would score very low. The honest path is learned features
  (CNN/ViT) with a symbolic head on top -- i.e. the FEATURE LAYER must
  itself be learned, which csp_sr does not provide.

So csp_sr is the strong, fast, interpretable CLASSIFIER; for 0.99 / large
images the FEATURE LAYER is the missing learned piece.